# OmniEvo - Quickstart

Este notebook demuestra el uso básico del paquete OmniEvo para optimización de atribución omnicanal.

## Problema
En marketing híbrido (digital + físico), determinar qué canal realmente impulsa el valor del cliente (LTV) es un desafío conocido como el **Problema de Atribución**.

## Solución
Usar un **Algoritmo Genético** para evolucionar pesos de atribución que minimicen el error entre el LTV predicho y el LTV real.

In [ ]:
# Instalación (ejecutar solo si es necesario)
# !pip install -e ..

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from omnievo import (
    DataGenerator,
    GeneticOptimizer,
    compare_baselines,
    plot_convergence,
    plot_weights,
    plot_comparison,
    plot_prediction_scatter,
    plot_evolution_dashboard,
)
from omnievo.fitness import predict_ltv

print("Librerías importadas correctamente")

## 1. Generación de Datos Sintéticos

Simulamos el "Smart Music Festival" con usuarios en diferentes segmentos.

In [ ]:
# Generar dataset
generator = DataGenerator(
    n_users=1000,
    noise_level=0.2,
    random_state=42,
)

df = generator.generate(normalize=True)
channels = generator.get_channel_names()

print(f"Dataset generado: {len(df)} usuarios")
print(f"Canales: {channels}")
print(f"\nDistribución de segmentos:")
print(df['segment'].value_counts())
df.head()

In [ ]:
# Separar features y target
X = df[channels].values
y = df['LTV_real'].values

# Split train/test (70/30)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f"Train set: {len(X_train)} usuarios")
print(f"Test set: {len(X_test)} usuarios")

## 2. Optimización con Algoritmo Genético

In [ ]:
# Configurar y ejecutar el AG
optimizer = GeneticOptimizer(
    population_size=50,
    generations=50,
    cxpb=0.7,
    mutpb=0.2,
    elite_size=2,
    random_state=42,
    verbose=True,
)

result = optimizer.fit(X_train, y_train)

print(f"\nOptimización completada")
print(f"Mejor fitness: {result.best_fitness:.4f}")
print(f"Convergió: {result.converged}")

In [ ]:
# Mostrar pesos optimizados
print("\nPesos de Atribución Optimizados:")
print("=" * 40)
weights_dict = result.get_weights_dict(channels)
for channel, weight in sorted(weights_dict.items(), key=lambda x: -x[1]):
    print(f"  {channel}: {weight:.4f} ({weight*100:.1f}%)")

## 3. Visualización de Resultados

In [ ]:
# Gráfica de convergencia
plot_convergence(result);

In [ ]:
# Gráfica de pesos
plot_weights(result.best_weights, channels);

## 4. Comparación con Modelos Baseline

In [ ]:
# Comparar en el test set
comparison = compare_baselines(
    X_test, y_test,
    ga_weights=result.best_weights,
    channel_names=channels,
)

print("Comparación de Modelos (Test Set)")
print("=" * 60)
comparison

In [ ]:
# Gráfica comparativa
plot_comparison(comparison);

In [ ]:
# Scatter plot de predicción
y_pred = predict_ltv(X_test, result.best_weights, scale_factor=y_test.max())
plot_prediction_scatter(y_test, y_pred);

## 5. Dashboard Completo

In [ ]:
# Dashboard con todas las visualizaciones
plot_evolution_dashboard(
    result=result,
    comparison_df=comparison,
    channel_names=channels,
    y_test=y_test,
    y_pred=y_pred,
);

## 6. Interpretación de Resultados

### Análisis de Negocio
Los pesos optimizados por el Algoritmo Genético revelan qué canales son más predictivos del LTV:

- **Canales IoT** (RFID, NFC): Típicamente tienen mayor peso porque capturan comportamiento real de consumo
- **Canales Digitales**: Contribuyen al awareness pero no siempre se correlacionan con el gasto final
- **Canales App**: Balance entre engagement digital y comportamiento transaccional

### Conclusiones
1. El AG supera a los modelos heurísticos tradicionales (Uniforme, Last-Touch)
2. La atribución data-driven es más precisa que las asignaciones arbitrarias
3. En contextos omnicanal, es crítico considerar múltiples touchpoints simultáneamente